# 05 RSU Intersection Stress Test

This notebook visualizes the second scenario (RSU at an intersection).

- Input:
  - `result/table/05_rsu_intersection_timeseries.csv`
  - `result/table/05_rsu_intersection_summary.csv`
- Output:
  - PDF vector figures in `result/figure/`
  - summary tables in `result/table/`


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os

# -----------------------------------------------------------------------------
# Fix: Handle paths for both Script (.py) and Jupyter Notebook (.ipynb)
# -----------------------------------------------------------------------------
try:
    # This works when running as a script
    current_path = Path(__file__).resolve()
    ROOT = current_path.parents[1]
except NameError:
    # This works in Jupyter Notebook
    current_path = Path.cwd()
    ROOT = current_path.parents[0]

# Verify the path is correct
print(f"Current ROOT: {ROOT}")

tbl_dir = ROOT / "result" / "table"
fig_dir = ROOT / "result" / "figure"
fig_dir.mkdir(parents=True, exist_ok=True)

# -----------------------------------------------------------------------------
# Load Data
# -----------------------------------------------------------------------------
timeseries_csv = tbl_dir / "05_rsu_intersection_timeseries.csv"
summary_csv = tbl_dir / "05_rsu_intersection_summary.csv"

if timeseries_csv.exists() and summary_csv.exists():
    df_ts = pd.read_csv(timeseries_csv)
    df_sum = pd.read_csv(summary_csv)
    methods = list(df_sum["method"].unique())
    print("Methods:", methods)
else:
    if not timeseries_csv.exists():
        print(f"Error: File not found at {timeseries_csv}")
    if not summary_csv.exists():
        print(f"Error: File not found at {summary_csv}")


In [ ]:
# Summary table: mean ± std across runs for each method
metrics = ["avg_sum_queue", "p95_max_queue", "avg_sense_u", "p95_sense_u",
           "alpha_total_variation", "switch_count", "deadline_violation_rate", "avg_slot_ms"]

agg = df_sum.groupby("method")[metrics].agg(["mean", "std"]).reset_index()
agg.columns = ["method"] + [f"{m}_{s}" for m in metrics for s in ["mean", "std"]]

out_table = tbl_dir / "05_rsu_intersection_table.csv"
agg.to_csv(out_table, index=False)
print("Saved table:", out_table)

agg


In [ ]:
# Time-series (mean across runs) for queue and sensing
g = df_ts.groupby(["method", "t"], as_index=False).agg(
    sum_queue=("sum_queue", "mean"),
    max_queue=("max_queue", "mean"),
    sense_u=("sense_u", "mean"),
    alpha=("alpha", "mean"),
)

def plot_metric(metric: str, ylabel: str, fname: str):
    plt.figure()
    for m in methods:
        sub = g[g["method"] == m]
        plt.plot(sub["t"], sub[metric], label=m)
    plt.xlabel("Slot $t$")
    plt.ylabel(ylabel)
    plt.legend()
    plt.grid(True, alpha=0.3)
    out = fig_dir / fname
    plt.savefig(out, format="pdf", bbox_inches="tight")
    print("Saved:", out)

plot_metric("sum_queue", "Average total queue", "05_rsu_timeseries_sum_queue.pdf")
plot_metric("sense_u", "Average sensing uncertainty $u_t$", "05_rsu_timeseries_sense_u.pdf")
plot_metric("alpha", "Average gate $\alpha_t$", "05_rsu_timeseries_alpha.pdf")


In [ ]:
# Empirical CDF of max queue (across all slots and runs)
plt.figure()
for m in methods:
    vals = df_ts[df_ts["method"] == m]["max_queue"].values
    xs = np.sort(vals)
    ys = np.linspace(0, 1, len(xs), endpoint=True)
    plt.plot(xs, ys, label=m)
plt.xlabel("Max queue length")
plt.ylabel("Empirical CDF")
plt.legend()
plt.grid(True, alpha=0.3)
out = fig_dir / "05_rsu_cdf_max_queue.pdf"
plt.savefig(out, format="pdf", bbox_inches="tight")
print("Saved:", out)


In [ ]:
# Boxplot: per-run avg slot time for each method
data = [df_sum[df_sum["method"] == m]["avg_slot_ms"].values for m in methods]
plt.figure()
plt.boxplot(data, labels=methods, vert=True, showfliers=False)
plt.ylabel("Avg per-slot runtime (ms)")
plt.xticks(rotation=30, ha="right")
plt.grid(True, axis="y", alpha=0.3)
out = fig_dir / "05_rsu_boxplot_slot_ms.pdf"
plt.savefig(out, format="pdf", bbox_inches="tight")
print("Saved:", out)
